# Tugas Akhir Kecerdasan Artifisial — Neural Network

## Prediksi Penyakit Jantung Menggunakan Artificial Neural Network (ANN)

**Dataset:** Heart Disease UCI (Kaggle)
**Fokus Studi:** Klasifikasi Penyakit Jantung
**Model:** Artificial Neural Network (ANN)

### Studi Kasus

Penyakit jantung merupakan salah satu penyebab kematian tertinggi di dunia. Pada penelitian ini digunakan dataset Heart Disease UCI untuk membangun model Artificial Neural Network (ANN) yang mampu memprediksi kemungkinan seseorang menderita penyakit jantung berdasarkan data medis pasien. Model dievaluasi menggunakan Accuracy, Precision, Recall, F1-Score, Confusion Matrix, dan ROC-AUC untuk mengukur performa klasifikasi.


In [ ]:
# ==========================================
# 1. IMPORT LIBRARY
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# ==========================================
# 3. LOAD DATASET
# ==========================================

file_id = "1PVbj-pbytZIr4pWGqiHAS0mkVHGwAD1a"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

df = pd.read_csv(url)

print("Shape Dataset :", df.shape)

display(df.head())

In [ ]:
# ==========================================
# 4. DATA UNDERSTANDING
# ==========================================

print("\nInfo Dataset")
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nStatistik Deskriptif")
display(df.describe())

In [ ]:
# ==========================================
# 5. DATA PREPROCESSING
# ==========================================

df = df.dropna()

# Hapus kolom yang tidak diperlukan
for col in ['id', 'dataset']:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

In [ ]:
# ==========================================
# 6. ENCODING DATA KATEGORIK
# ==========================================

categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
# ==========================================
# 7. MEMBUAT TARGET BINER
# ==========================================

if 'num' in df.columns:
    df['target'] = np.where(df['num'] > 0, 1, 0)
    df.drop('num', axis=1, inplace=True)

In [ ]:
# ==========================================
# 8. FEATURE DAN TARGET
# ==========================================

X = df.drop('target', axis=1)
y = df['target']

print("\nJumlah Fitur :", X.shape[1])

In [ ]:
# ==========================================
# 9. TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# ==========================================
# 10. NORMALISASI DATA
# ==========================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# ==========================================
# 11. CLASS WEIGHT
# ==========================================

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight_dict = {
    0: weights[0],
    1: weights[1]
}

print("\nClass Weight")
print(class_weight_dict)

In [ ]:
# ==========================================
# 12. MEMBANGUN NEURAL NETWORK
# ==========================================

model = Sequential()

model.add(
    Dense(
        32,
        activation='relu',
        input_shape=(X_train.shape[1],)
    )
)

model.add(Dropout(0.3))

model.add(
    Dense(
        16,
        activation='relu'
    )
)

model.add(Dropout(0.2))

model.add(
    Dense(
        8,
        activation='relu'
    )
)

model.add(Dropout(0.1))

model.add(
    Dense(
        1,
        activation='sigmoid'
    )
)

In [ ]:
# ==========================================
# 13. COMPILE MODEL
# ==========================================

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nModel Summary")
model.summary()

In [ ]:
# ==========================================
# 14. EARLY STOPPING
# ==========================================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
# ==========================================
# 15. TRAINING MODEL
# ==========================================

history = model.fit(
    X_train,
    y_train,
    epochs=200,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop],
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
# ==========================================
# 16. EVALUASI MODEL
# ==========================================

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\n========================")
print("HASIL EVALUASI MODEL")
print("========================")
print("Loss     :", round(loss, 4))
print("Accuracy :", round(accuracy, 4))

In [ ]:
# ==========================================
# 17. PREDIKSI
# ==========================================

y_pred_prob = model.predict(X_test)

# Threshold lebih sensitif
y_pred = (y_pred_prob > 0.4).astype(int)

In [ ]:
# ==========================================
# 18. METRIK EVALUASI
# ==========================================

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n========================")
print("METRIK EVALUASI")
print("========================")
print("Accuracy  :", round(acc, 4))
print("Precision :", round(prec, 4))
print("Recall    :", round(rec, 4))
print("F1 Score  :", round(f1, 4))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

In [ ]:
# ==========================================
# 19. CONFUSION MATRIX
# ==========================================

cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
# ==========================================
# 20. ROC CURVE DAN AUC
# ==========================================

fpr, tpr, thresholds = roc_curve(
    y_test,
    y_pred_prob
)

roc_auc = auc(
    fpr,
    tpr
)

plt.figure(figsize=(7,5))

plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f'AUC = {roc_auc:.4f}'
)

plt.plot(
    [0,1],
    [0,1],
    linestyle='--'
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()

print("AUC Score :", round(roc_auc, 4))

In [ ]:
# ==========================================
# 21. GRAFIK LOSS
# ==========================================

plt.figure(figsize=(8,5))

plt.plot(
    history.history['loss'],
    label='Training Loss'
)

plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

In [ ]:
# ==========================================
# 22. GRAFIK ACCURACY
# ==========================================

plt.figure(figsize=(8,5))

plt.plot(
    history.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()